## 1. Parâmetros do ambiente

In [0]:
catalog = "cinedata_analytics"
bronze_schema_name = "bronze"
silver_schema_name = "silver"
gold_schema_name = "gold"

bronze_schema = f"{catalog}.{bronze_schema_name}"
silver_schema = f"{catalog}.{silver_schema_name}"
gold_schema = f"{catalog}.{gold_schema_name}"

landing_path = f"/Volumes/{catalog}/{bronze_schema_name}/landing"

print(f"catalog: {catalog}")
print(f"bronze_schema: {bronze_schema}")
print(f"silver_schema: {silver_schema}")
print(f"gold_schema: {gold_schema}")
print(f"landing_path: {landing_path}\n")

catalog: cinedata_analytics
bronze_schema: cinedata_analytics.bronze
silver_schema: cinedata_analytics.silver
gold_schema: cinedata_analytics.gold
landing_path: /Volumes/cinedata_analytics/bronze/landing



## 2. Criação do Catalog, Schemas e Volume


In [0]:
spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {bronze_schema}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {silver_schema}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {gold_schema}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {bronze_schema}.landing")

print("✅ Catalog, schemas e volume prontos.\n")

✅ Catalog, schemas e volume prontos.



## 3. Validação da Landing Zone

In [0]:
expected_files = [
    "credits_and_tags_IMDB_TMDB.csv",
    "movies_financials_IMDB_TMDB.csv",
    "movies_info_TMDB_IMDB.csv",
    "movies_metrics_IMDB_TMDB.csv",
    "movies_reviews.csv"
]

try:
    existing = {f.name for f in dbutils.fs.ls(landing_path)}
except Exception as e:
    existing = set()
    print(f"[ALERTA] Não foi possível listar '{landing_path}'. Faça o upload dos arquivos pela interface do Catalog antes de continuar.\nErro: {e}")

missing = [f for f in expected_files if f not in existing]

if missing:
    print("❌ [PENDENTE] Arquivos ainda não encontrados na landing zone (Faça o upload):")
    for f in missing:
        print(f"  - {f}")
else:
    print("🚀 [OK] Todos os arquivos esperados estão na landing zone:")
    for f in dbutils.fs.ls(landing_path):
        print(f"  - {f.name} ({f.size/1024/1024:.2f} MB)")

🚀 [OK] Todos os arquivos esperados estão na landing zone:
  - credits_and_tags_IMDB_TMDB.csv (21.29 MB)
  - movies_financials_IMDB_TMDB.csv (1.40 MB)
  - movies_info_TMDB_IMDB.csv (32.35 MB)
  - movies_metrics_IMDB_TMDB.csv (3.17 MB)
  - movies_reviews.csv (1.83 MB)


## 4. Ingestão de Dados na Bronze

### Architecture Decision Record (ADR): Tratamento de I/O na Ingestão (multiLine e escape)

**Contexto e Decisão:**
Para garantir o princípio arquitetural da camada Bronze de "evitar perda de dados na origem", as opções `.option("multiLine", "true")` e `.option("escape", '"')` estão ativas para **todos** os arquivos CSV. O objetivo é blindar o parser do Spark contra o *Column Shift* (deslocamento de colunas) causado por quebras de linha irregulares e aspas aninhadas em campos de texto livre.

**Comprovação Empírica (Testes de I/O):**
A execução de quatro cenários de leitura comprovou o comportamento destrutivo da origem caso as opções fossem desativadas:

| Arquivo (Tabela) | Nenhuma Option | Apenas `escape='"'` | Apenas `multiLine="true"` | Ambas as Options Ativas |
| :--- | :--- | :--- | :--- | :--- |
| **movies_info** | 106.930 | 106.930 | 101.571 | **106.596** |
| **movies_financials** | 106.165 | 106.165 | 106.165 | **106.165** |
| **movies_metrics** | 107.364 | 107.364 | 103.214 | **103.072** |
| **credits_and_tags** | 106.320 | 106.320 | 105.679 | **105.608** |
| **movies_reviews** | 32.412 | 32.412 | 32.412 | **32.412** |

**Justificativa Técnica:**
1. **Sem proteção (Inflação e Corrupção):** O Spark quebra os registros a cada `\n` acidental no meio de um texto, inflando a base artificialmente com "linhas fantasmas" e deslocando as colunas à direita.
2. **Apenas `multiLine` (Perda Silenciosa):** O Spark se perde em aspas duplas aninhadas (`""`), assumindo finais de texto incorretos e "engolindo" milhares de registros ao aglutinar múltiplos filmes numa única linha (ex: *movies_info* despencou para 101.571).
3. **Aplicação em Todas as Tabelas:** Tabelas teóricamente numéricas (como *movies_metrics*) sofreram variações severas, provando que a sujeira textual já vazou para colunas numéricas. Além disso, arquivos como *movies_reviews* possuem colunas de *Free-Text* de preenchimento humano. Manter a blindagem para todos os arquivos garante a integridade total da camada Bronze e previne a quebra do pipeline por viés de amostragem.

In [0]:
import pyspark.sql.functions as F

mapeamento_arquivos = {
    "movies_info_TMDB_IMDB.csv": "tb_movies_info",
    "movies_financials_IMDB_TMDB.csv": "tb_movies_financials",
    "movies_metrics_IMDB_TMDB.csv": "tb_movies_metrics",
    "credits_and_tags_IMDB_TMDB.csv": "tb_credits_and_tags",
    "movies_reviews.csv": "tb_movies_reviews"
}

print("Iniciando ingestão para a Camada Bronze...\n")

for arquivo, nome_tabela in mapeamento_arquivos.items():
    caminho_arquivo = f"{landing_path}/{arquivo}"
    tabela_destino = f"{bronze_schema}.{nome_tabela}"
    
    df_raw = (
        spark.read.format("csv")
        .option("header", "true")
        .option("multiLine", "true")
        .option("escape", '"')
        .load(caminho_arquivo)
    ) 
    
    (
        df_raw
        .withColumn("ingestion_datetime", F.current_timestamp())
        .write
        .format("delta")
        .mode("append")
        .saveAsTable(tabela_destino)
    )
    
    print(f"✅ [SUCESSO] Ingestão concluída: {arquivo} -> {tabela_destino}")
    print(f"Quantidade de linhas: {df_raw.count()}")


Iniciando ingestão para a Camada Bronze...

✅ [SUCESSO] Ingestão concluída: movies_info_TMDB_IMDB.csv -> cinedata_analytics.bronze.tb_movies_info
Quantidade de linhas: 101571
✅ [SUCESSO] Ingestão concluída: movies_financials_IMDB_TMDB.csv -> cinedata_analytics.bronze.tb_movies_financials
Quantidade de linhas: 106165
✅ [SUCESSO] Ingestão concluída: movies_metrics_IMDB_TMDB.csv -> cinedata_analytics.bronze.tb_movies_metrics
Quantidade de linhas: 103214
✅ [SUCESSO] Ingestão concluída: credits_and_tags_IMDB_TMDB.csv -> cinedata_analytics.bronze.tb_credits_and_tags
Quantidade de linhas: 105679
✅ [SUCESSO] Ingestão concluída: movies_reviews.csv -> cinedata_analytics.bronze.tb_movies_reviews
Quantidade de linhas: 32412


## 5. Ingestão de Dados via API (Cotação do Dólar)

**Parametrização via Widgets com Fallback Dinâmico:**

A ingestão utiliza `dbutils.widgets` para permitir controle manual do período de extração (`data_inicio` e `data_fim`), mas os widgets são criados **vazios** (`""`) propositalmente. Quando não preenchidos, o código calcula automaticamente os **últimos 7 dias** a partir da data atual. Esta estratégia garante:

- **Automação:** Execuções agendadas (Jobs) funcionam sem intervenção manual, alimentando-se do fallback dinâmico.
- **Controle manual:** Um operador pode inserir datas específicas nos widgets para extrações pontuais ou reprocessamentos históricos.
- **Sem estado preso:** Ao não pré-popular os widgets com valores padrão, evita-se o risco de execuções futuras reutilizarem datas obsoletas deixadas em sessões anteriores.

In [0]:
import requests
from datetime import datetime, timedelta
import pyspark.sql.functions as F

dbutils.widgets.text("data_inicio", "", "Data Início (MM-DD-AAAA)")
dbutils.widgets.text("data_fim", "", "Data Fim (MM-DD-AAAA)")

param_inicio = dbutils.widgets.get("data_inicio").strip()
param_fim = dbutils.widgets.get("data_fim").strip()

if param_inicio == "" or param_fim == "":
    print("⚠️ Widgets vazios. Calculando dinamicamente os últimos 7 dias...")
    hoje = datetime.now()
    sete_dias_atras = hoje - timedelta(days=7)
    
    data_inicio_formatada = sete_dias_atras.strftime("%m-%d-%Y")
    data_fim_formatada = hoje.strftime("%m-%d-%Y")
else:
    print("📌 Utilizando as datas inseridas manualmente nos Widgets...")
    data_inicio_formatada = param_inicio
    data_fim_formatada = param_fim

print(f"Extraindo cotação do período: {data_inicio_formatada} a {data_fim_formatada}...")

url = f"https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata/CotacaoDolarPeriodo(dataInicial=@dataInicial,dataFinalCotacao=@dataFinalCotacao)?@dataInicial='{data_inicio_formatada}'&@dataFinalCotacao='{data_fim_formatada}'&$select=dataHoraCotacao,cotacaoCompra&$format=json"

response = requests.get(url)

if response.status_code == 200:
    dados_json = response.json().get("value", [])
    
    if not dados_json:
        print("A API não retornou dados. Possível fim de semana ou feriado.")
    else:
        df_api_raw = spark.createDataFrame(dados_json)
        tabela_destino = f"{bronze_schema}.tb_cotacao_dolar"
        
        (
            df_api_raw
            .withColumn("ingestion_datetime", F.current_timestamp())
            .write
            .format("delta")
            .mode("append")
            .saveAsTable(tabela_destino)
        )
        
        print(f"✅ [SUCESSO] Cotação do Dólar salva em: {tabela_destino}")
        display(spark.table(tabela_destino).limit(5))
        
else:
    print(f"❌ [ERRO] Conexão com API falhou. HTTP Status: {response.status_code}")

⚠️ Widgets vazios. Calculando dinamicamente os últimos 7 dias...
Extraindo cotação do período: 09-14-2026 a 09-21-2026...
✅ [SUCESSO] Cotação do Dólar salva em: cinedata_analytics.bronze.tb_cotacao_dolar


cotacaoCompra,dataHoraCotacao,ingestion_datetime
5.169,2026-09-14 13:10:08.144425,2026-09-20T03:50:47.246Z
5.1484,2026-09-15 13:09:19.199664,2026-09-20T03:50:47.246Z
5.152,2026-09-16 13:05:30.35873,2026-09-20T03:50:47.246Z
5.1515,2026-09-17 13:03:21.858212,2026-09-20T03:50:47.246Z
5.1569,2026-09-18 13:03:34.742036,2026-09-20T03:50:47.246Z
